In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# =============================================================================
# Supermarket Sales Analysis — Data Analytics Project
# Author : Manthan
# Date   : September 2026
# Dataset: SUPER MARKET DATA.xlsx (500 transactions)
# =============================================================================

# 🛒 Supermarket Sales Analysis
## Data Analytics Project — by Manthan

**Objective:** Analyze supermarket sales data to extract actionable insights
about products, branches, categories, customers, payment methods, and ratings.

---
## 1. Imports & Setup

In [2]:
# --- Core Libraries ---
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --- Visualization Style ---
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#ffffff',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.3,
})

# Color palette
COLORS = {
    'primary': '#4361ee',
    'secondary': '#3a0ca3',
    'accent': '#f72585',
    'success': '#06d6a0',
    'warning': '#ffd166',
    'info': '#118ab2',
    'dark': '#073b4c',
}
PALETTE_MAIN = ['#4361ee', '#f72585', '#06d6a0', '#ffd166', '#118ab2',
                '#3a0ca3', '#7209b7', '#e63946']
PALETTE_SEQ = 'coolwarm'

# Charts output directory
CHART_DIR = 'charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_chart(fig, name):
    """Save a chart to the charts directory."""
    path = os.path.join(CHART_DIR, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'  ✅ Saved: {path}')
    return path

print('✅ Libraries imported & style configured.')

✅ Libraries imported & style configured.


---
## 2. Data Loading

In [3]:
# Load dataset
DATA_FILE = 'SUPER MARKET DATA.xlsx'
df = pd.read_excel(DATA_FILE)

print(f'📊 Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns\n')
print('--- First 5 Rows ---')
print(df.head().to_string())
print('\n--- Column Data Types ---')
print(df.dtypes)
print(f'\n--- Dataset Info ---')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

📊 Dataset loaded: 500 rows × 13 columns

--- First 5 Rows ---
  Invoice ID       Date Branch    City Customer Type  Gender  Product       Category  Quantity  Unit Price      Payment  Rating   Sales
0    INV0001 2026-06-01      A  Jaipur        Member    Male     Milk          Dairy         4       57.13          UPI     3.8  228.52
1    INV0002 2026-02-26      A  Jaipur        Normal    Male     Rice        Grocery         9       66.85  Net Banking     3.9  601.65
2    INV0003 2026-02-09      C  Mumbai        Normal  Female     Milk          Dairy         3       57.28         Card     3.4  171.84
3    INV0004 2026-01-12      C  Mumbai        Member  Female  Shampoo  Personal Care         2      168.27         Card     4.5  336.54
4    INV0005 2026-06-30      A  Jaipur        Normal  Female   Apples         Fruits         5      130.57         Cash     3.1  652.85

--- Column Data Types ---
Invoice ID                  str
Date             datetime64[us]
Branch                      str

---
## 3. Data Cleaning & Preprocessing

In [4]:
print('🔍 DATA QUALITY CHECKS\n')

# 3.1 Missing values
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# 3.2 Duplicate rows
duplicates = df.duplicated().sum()
print(f'\n--- Duplicate Rows: {duplicates} ---')

# 3.3 Verify Sales = Quantity × Unit Price
df['Sales_Calculated'] = (df['Quantity'] * df['Unit Price']).round(2)
mismatch = df[df['Sales'] != df['Sales_Calculated']]
print(f'\n--- Sales Verification ---')
print(f'Mismatches between Sales and Quantity × Unit Price: {len(mismatch)}')
if len(mismatch) > 0:
    print('Fixing mismatched rows...')
    df['Sales'] = df['Sales_Calculated']
df.drop(columns=['Sales_Calculated'], inplace=True)

# 3.4 Data type checks
print('\n--- Data Type Validation ---')
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Quantity range: {df['Quantity'].min()} to {df['Quantity'].max()}")
print(f"Unit Price range: ₹{df['Unit Price'].min():.2f} to ₹{df['Unit Price'].max():.2f}")
print(f"Sales range: ₹{df['Sales'].min():.2f} to ₹{df['Sales'].max():.2f}")
print(f"Rating range: {df['Rating'].min()} to {df['Rating'].max()}")

# 3.5 Add derived columns for time-based analysis
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%B')
df['Day_of_Week'] = df['Date'].dt.day_name()

print('\n✅ Data cleaning complete. Added Month, Month_Name, Day_of_Week columns.')

🔍 DATA QUALITY CHECKS

--- Missing Values ---
Invoice ID       0
Date             0
Branch           0
City             0
Customer Type    0
Gender           0
Product          0
Category         0
Quantity         0
Unit Price       0
Payment          0
Rating           0
Sales            0
dtype: int64

Total missing values: 0

--- Duplicate Rows: 0 ---

--- Sales Verification ---
Mismatches between Sales and Quantity × Unit Price: 0

--- Data Type Validation ---
Date range: 2026-01-01 to 2026-07-01
Quantity range: 1 to 10
Unit Price range: ₹27.67 to ₹236.71
Sales range: ₹28.61 to ₹2114.82
Rating range: 3.0 to 5.0

✅ Data cleaning complete. Added Month, Month_Name, Day_of_Week columns.


---
## 4. Descriptive Statistics

In [5]:
print('📈 DESCRIPTIVE STATISTICS\n')
print(df.describe().round(2).to_string())

print('\n--- Categorical Column Summary ---')
for col in ['Branch', 'City', 'Customer Type', 'Gender', 'Category', 'Payment']:
    print(f"\n{col}: {df[col].nunique()} unique → {df[col].value_counts().to_dict()}")

📈 DESCRIPTIVE STATISTICS

                             Date  Quantity  Unit Price  Rating    Sales   Month
count                         500    500.00      500.00  500.00   500.00  500.00
mean   2026-04-03 12:25:55.200000      5.54       88.66    3.99   488.82    3.59
min           2026-01-01 00:00:00      1.00       27.67    3.00    28.61    1.00
25%           2026-02-17 00:00:00      3.00       43.96    3.50   189.21    2.00
50%           2026-04-09 00:00:00      5.50       61.88    4.00   335.91    4.00
75%           2026-05-17 06:00:00      8.00      128.57    4.50   609.88    5.00
max           2026-07-01 00:00:00     10.00      236.71    5.00  2114.82    7.00
std                           NaN      2.88       57.28    0.57   437.33    1.75

--- Categorical Column Summary ---

Branch: 4 unique → {'C': 143, 'B': 133, 'D': 119, 'A': 105}

City: 4 unique → {'Mumbai': 143, 'Delhi': 133, 'Bengaluru': 119, 'Jaipur': 105}

Customer Type: 2 unique → {'Member': 296, 'Normal': 204}

Gender: 

---
## 5. EDA — Product Analysis

*Which product generates the highest sales?*

In [6]:
print('🛍️  PRODUCT ANALYSIS\n')

# Total sales by product
product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=False).round(2)
print('--- Total Sales by Product (Top 10) ---')
print(product_sales.head(10).to_string())
print(f'\n🏆 Highest: {product_sales.index[0]} — ₹{product_sales.iloc[0]:,.2f}')
print(f'📉 Lowest : {product_sales.index[-1]} — ₹{product_sales.iloc[-1]:,.2f}')

# --- Chart: Top 10 Products by Sales ---
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(product_sales.index[::-1], product_sales.values[::-1],
               color=sns.color_palette('coolwarm_r', n_colors=len(product_sales)),
               edgecolor='white', linewidth=0.5)
# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width + 200, bar.get_y() + bar.get_height()/2,
            f'₹{width:,.0f}', ha='left', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Total Sales (₹)')
ax.set_title('Total Sales by Product', fontsize=16, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_chart(fig, '01_product_sales')

# --- Chart: Average Sales per Transaction by Product ---
product_avg = df.groupby('Product')['Sales'].mean().sort_values(ascending=False).round(2)
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(product_avg.index[::-1], product_avg.values[::-1],
               color=sns.color_palette('mako', n_colors=len(product_avg)),
               edgecolor='white', linewidth=0.5)
for bar in bars:
    width = bar.get_width()
    ax.text(width + 10, bar.get_y() + bar.get_height()/2,
            f'₹{width:,.0f}', ha='left', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Average Sales per Transaction (₹)')
ax.set_title('Average Sales per Transaction by Product', fontsize=16, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_chart(fig, '02_product_avg_sales')

🛍️  PRODUCT ANALYSIS

--- Total Sales by Product (Top 10) ---
Product
Cheese         27906.30
Coffee         27694.87
Shampoo        27497.48
Cooking Oil    21525.06
Tea            17681.59
Apples         17057.41
Face Wash      11946.85
Rice           11754.17
Cold Drink     10731.78
Eggs           10429.77

🏆 Highest: Cheese — ₹27,906.30
📉 Lowest : Biscuits — ₹3,908.87


  ✅ Saved: charts\01_product_sales.png


  ✅ Saved: charts\02_product_avg_sales.png


'charts\\02_product_avg_sales.png'

---
## 6. EDA — Branch & City Analysis

*Which branch performs best?*

In [7]:
print('🏢 BRANCH & CITY ANALYSIS\n')

# Sales by branch
branch_sales = df.groupby(['Branch', 'City'])['Sales'].agg(['sum', 'mean', 'count']).round(2)
branch_sales.columns = ['Total Sales', 'Avg Transaction', 'Transactions']
branch_sales = branch_sales.sort_values('Total Sales', ascending=False)
print('--- Sales by Branch & City ---')
print(branch_sales.to_string())

best_branch = branch_sales['Total Sales'].idxmax()
print(f'\n🏆 Best Branch: {best_branch[0]} ({best_branch[1]}) — ₹{branch_sales.loc[best_branch, "Total Sales"]:,.2f}')

# --- Chart: Branch Comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Branch Performance Comparison', fontsize=18, fontweight='bold', y=1.02)

branch_data = df.groupby('Branch').agg(
    Total_Sales=('Sales', 'sum'),
    Avg_Transaction=('Sales', 'mean'),
    Transactions=('Sales', 'count')
).reset_index()
branch_data['City'] = branch_data['Branch'].map(
    df.groupby('Branch')['City'].first().to_dict()
)
branch_data['Label'] = branch_data['Branch'] + '\n(' + branch_data['City'] + ')'

# Total Sales
bars1 = axes[0].bar(branch_data['Label'], branch_data['Total_Sales'],
                     color=PALETTE_MAIN[:4], edgecolor='white', linewidth=1.5)
axes[0].set_title('Total Sales', fontweight='bold')
axes[0].set_ylabel('Sales (₹)')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Avg Transaction
bars2 = axes[1].bar(branch_data['Label'], branch_data['Avg_Transaction'],
                     color=PALETTE_MAIN[:4], edgecolor='white', linewidth=1.5)
axes[1].set_title('Avg Transaction Value', fontweight='bold')
axes[1].set_ylabel('Avg Sales (₹)')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Transaction Count
bars3 = axes[2].bar(branch_data['Label'], branch_data['Transactions'],
                     color=PALETTE_MAIN[:4], edgecolor='white', linewidth=1.5)
axes[2].set_title('Number of Transactions', fontweight='bold')
axes[2].set_ylabel('Count')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '03_branch_comparison')

🏢 BRANCH & CITY ANALYSIS

--- Sales by Branch & City ---
                  Total Sales  Avg Transaction  Transactions
Branch City                                                 
C      Mumbai        72469.45           506.78           143
B      Delhi         64116.26           482.08           133
D      Bengaluru     55468.29           466.12           119
A      Jaipur        52357.08           498.64           105

🏆 Best Branch: C (Mumbai) — ₹72,469.45


  ✅ Saved: charts\03_branch_comparison.png


'charts\\03_branch_comparison.png'

---
## 7. EDA — Category Analysis

*Which category sells the most?*

In [8]:
print('📦 CATEGORY ANALYSIS\n')

# Sales by category
cat_sales = df.groupby('Category')['Sales'].agg(['sum', 'mean', 'count']).round(2)
cat_sales.columns = ['Total Sales', 'Avg Transaction', 'Transactions']
cat_sales = cat_sales.sort_values('Total Sales', ascending=False)
print('--- Sales by Category ---')
print(cat_sales.to_string())
print(f'\n🏆 Top Category: {cat_sales.index[0]} — ₹{cat_sales.iloc[0]["Total Sales"]:,.2f}')

# --- Chart: Category Sales ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Category Analysis', fontsize=18, fontweight='bold', y=1.02)

# Total sales
colors_cat = sns.color_palette('rocket', n_colors=len(cat_sales))
bars = axes[0].barh(cat_sales.index[::-1], cat_sales['Total Sales'].values[::-1],
                     color=colors_cat, edgecolor='white', linewidth=0.5)
for bar in bars:
    width = bar.get_width()
    axes[0].text(width + 300, bar.get_y() + bar.get_height()/2,
                 f'₹{width:,.0f}', ha='left', va='center', fontsize=9, fontweight='bold')
axes[0].set_title('Total Sales by Category', fontweight='bold')
axes[0].set_xlabel('Total Sales (₹)')
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Transaction count pie
wedges, texts, autotexts = axes[1].pie(
    cat_sales['Transactions'], labels=cat_sales.index, autopct='%1.1f%%',
    colors=sns.color_palette('Set2', n_colors=len(cat_sales)),
    startangle=140, pctdistance=0.8,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')
axes[1].set_title('Transaction Share by Category', fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '04_category_analysis')

📦 CATEGORY ANALYSIS

--- Sales by Category ---
               Total Sales  Avg Transaction  Transactions
Category                                                 
Beverages         56108.24           684.25            82
Personal Care     45943.96           629.37            73
Dairy             43992.00           646.94            68
Grocery           40470.47           493.54            82
Fruits            23263.17           528.71            44
Snacks            16992.97           226.57            75
Vegetables        11124.17           231.75            48
Bakery             6516.10           232.72            28

🏆 Top Category: Beverages — ₹56,108.24


  ✅ Saved: charts\04_category_analysis.png


'charts\\04_category_analysis.png'

---
## 8. EDA — Payment Method Analysis

*What is the most popular payment method?*

In [9]:
print('💳 PAYMENT METHOD ANALYSIS\n')

# Payment analysis
payment_stats = df.groupby('Payment')['Sales'].agg(['count', 'sum', 'mean']).round(2)
payment_stats.columns = ['Transactions', 'Total Sales', 'Avg Transaction']
payment_stats = payment_stats.sort_values('Transactions', ascending=False)
print('--- Payment Method Statistics ---')
print(payment_stats.to_string())
print(f'\n🏆 Most Popular: {payment_stats.index[0]} — {payment_stats.iloc[0]["Transactions"]} transactions')

# --- Chart: Payment Methods ---
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
fig.suptitle('Payment Method Analysis', fontsize=18, fontweight='bold', y=1.02)

# Transaction count — Donut chart
colors_pay = [COLORS['primary'], COLORS['accent'], COLORS['success'], COLORS['warning']]
wedges, texts, autotexts = axes[0].pie(
    payment_stats['Transactions'], labels=payment_stats.index,
    autopct='%1.1f%%', colors=colors_pay, startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
axes[0].set_title('Transaction Count by Payment', fontweight='bold')
# Add center text
centre_circle = plt.Circle((0, 0), 0.30, fc='white')
axes[0].add_artist(centre_circle)
axes[0].text(0, 0, f'{payment_stats["Transactions"].sum()}\nTotal', ha='center',
             va='center', fontsize=13, fontweight='bold', color=COLORS['dark'])

# Total sales by payment
bars = axes[1].bar(payment_stats.index, payment_stats['Total Sales'],
                    color=colors_pay, edgecolor='white', linewidth=1.5)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')
axes[1].set_title('Total Sales by Payment Method', fontweight='bold')
axes[1].set_ylabel('Total Sales (₹)')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '05_payment_analysis')

💳 PAYMENT METHOD ANALYSIS

--- Payment Method Statistics ---
             Transactions  Total Sales  Avg Transaction
Payment                                                
UPI                   127     67910.33           534.73
Net Banking           126     65194.93           517.42
Card                  125     57265.64           458.13
Cash                  122     54040.18           442.95

🏆 Most Popular: UPI — 127.0 transactions


  ✅ Saved: charts\05_payment_analysis.png


'charts\\05_payment_analysis.png'

---
## 9. EDA — Customer Type Analysis

*Do Members spend more than Normal customers?*

In [10]:
print('👥 CUSTOMER TYPE ANALYSIS\n')

# Customer type comparison
cust_stats = df.groupby('Customer Type')['Sales'].agg(['count', 'sum', 'mean']).round(2)
cust_stats.columns = ['Transactions', 'Total Sales', 'Avg Transaction']
print('--- Customer Type Comparison ---')
print(cust_stats.to_string())

member_avg = cust_stats.loc['Member', 'Avg Transaction']
normal_avg = cust_stats.loc['Normal', 'Avg Transaction']
print(f'\nMember Avg: ₹{member_avg:,.2f}  |  Normal Avg: ₹{normal_avg:,.2f}')
if member_avg > normal_avg:
    print('→ Members spend MORE per transaction.')
else:
    print('→ Normal customers spend MORE per transaction.')

# --- Chart: Customer Type ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Customer Type Analysis: Member vs Normal', fontsize=18, fontweight='bold', y=1.02)

cust_colors = [COLORS['primary'], COLORS['accent']]

# Transactions
bars1 = axes[0].bar(cust_stats.index, cust_stats['Transactions'], color=cust_colors,
                     edgecolor='white', linewidth=1.5)
axes[0].set_title('Transaction Count', fontweight='bold')
axes[0].set_ylabel('Count')
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Total Sales
bars2 = axes[1].bar(cust_stats.index, cust_stats['Total Sales'], color=cust_colors,
                     edgecolor='white', linewidth=1.5)
axes[1].set_title('Total Sales', fontweight='bold')
axes[1].set_ylabel('Sales (₹)')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Avg Transaction
bars3 = axes[2].bar(cust_stats.index, cust_stats['Avg Transaction'], color=cust_colors,
                     edgecolor='white', linewidth=1.5)
axes[2].set_title('Average Transaction Value', fontweight='bold')
axes[2].set_ylabel('Avg Sales (₹)')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '06_customer_type_analysis')

👥 CUSTOMER TYPE ANALYSIS

--- Customer Type Comparison ---
               Transactions  Total Sales  Avg Transaction
Customer Type                                            
Member                  296    143009.30           483.14
Normal                  204    101401.78           497.07

Member Avg: ₹483.14  |  Normal Avg: ₹497.07
→ Normal customers spend MORE per transaction.


  ✅ Saved: charts\06_customer_type_analysis.png


'charts\\06_customer_type_analysis.png'

---
## 10. EDA — Gender Analysis

In [11]:
print('🧑‍🤝‍🧑 GENDER ANALYSIS\n')

# Gender breakdown
gender_stats = df.groupby('Gender')['Sales'].agg(['count', 'sum', 'mean']).round(2)
gender_stats.columns = ['Transactions', 'Total Sales', 'Avg Transaction']
print('--- Gender Comparison ---')
print(gender_stats.to_string())

# Gender × Category
gender_cat = df.groupby(['Gender', 'Category'])['Sales'].sum().unstack(fill_value=0).round(2)
print('\n--- Sales by Gender × Category ---')
print(gender_cat.to_string())

# --- Chart: Gender Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Gender-Based Sales Analysis', fontsize=18, fontweight='bold', y=1.02)

gender_colors = ['#4361ee', '#f72585']

# Total sales by gender
bars = axes[0].bar(gender_stats.index, gender_stats['Total Sales'], color=gender_colors,
                    edgecolor='white', linewidth=1.5, width=0.5)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Total Sales by Gender', fontweight='bold')
axes[0].set_ylabel('Total Sales (₹)')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Gender × Category grouped bar
gender_cat.T.plot(kind='bar', ax=axes[1], color=gender_colors, edgecolor='white', linewidth=1)
axes[1].set_title('Sales by Category & Gender', fontweight='bold')
axes[1].set_ylabel('Total Sales (₹)')
axes[1].set_xlabel('Category')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
axes[1].legend(title='Gender')
axes[1].tick_params(axis='x', rotation=45)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '07_gender_analysis')

🧑‍🤝‍🧑 GENDER ANALYSIS

--- Gender Comparison ---
        Transactions  Total Sales  Avg Transaction
Gender                                            
Female           255    123954.12           486.09
Male             245    120456.96           491.66

--- Sales by Gender × Category ---
Category   Bakery  Beverages     Dairy    Fruits   Grocery  Personal Care   Snacks  Vegetables
Gender                                                                                        
Female    3452.47   24763.51  23683.89  13511.02  20974.20       23791.00  7114.17     6663.86
Male      3063.63   31344.73  20308.11   9752.15  19496.27       22152.96  9878.80     4460.31


  ✅ Saved: charts\07_gender_analysis.png


'charts\\07_gender_analysis.png'

---
## 11. EDA — Rating Analysis

*What is the average customer rating?*

In [12]:
print('⭐ RATING ANALYSIS\n')

avg_rating = df['Rating'].mean()
print(f'Average Customer Rating: {avg_rating:.2f} / 5.0\n')

# Rating by branch
rating_branch = df.groupby('Branch')['Rating'].mean().round(2)
print('--- Average Rating by Branch ---')
print(rating_branch.to_string())

# Rating by category
rating_cat = df.groupby('Category')['Rating'].mean().sort_values(ascending=False).round(2)
print('\n--- Average Rating by Category ---')
print(rating_cat.to_string())

# --- Chart: Rating Distribution & Analysis ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Rating Analysis', fontsize=18, fontweight='bold', y=1.01)

# Distribution histogram
axes[0, 0].hist(df['Rating'], bins=20, color=COLORS['primary'], edgecolor='white',
                linewidth=0.8, alpha=0.85)
axes[0, 0].axvline(avg_rating, color=COLORS['accent'], linestyle='--', linewidth=2,
                    label=f'Mean: {avg_rating:.2f}')
axes[0, 0].set_title('Rating Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Rating by branch
bars = axes[0, 1].bar(rating_branch.index, rating_branch.values, color=PALETTE_MAIN[:4],
                       edgecolor='white', linewidth=1.5)
for bar in bars:
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0, 1].set_title('Avg Rating by Branch', fontweight='bold')
axes[0, 1].set_ylabel('Average Rating')
axes[0, 1].set_ylim(0, 5.5)

# Rating by category
bars = axes[1, 0].barh(rating_cat.index[::-1], rating_cat.values[::-1],
                        color=sns.color_palette('viridis', n_colors=len(rating_cat)),
                        edgecolor='white', linewidth=0.5)
for bar in bars:
    axes[1, 0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.2f}', ha='left', va='center', fontsize=9, fontweight='bold')
axes[1, 0].set_title('Avg Rating by Category', fontweight='bold')
axes[1, 0].set_xlabel('Average Rating')
axes[1, 0].set_xlim(0, 5.5)

# Rating vs Sales scatter
scatter = axes[1, 1].scatter(df['Rating'], df['Sales'], alpha=0.5,
                              c=df['Rating'], cmap='coolwarm', edgecolors='white',
                              linewidth=0.3, s=50)
axes[1, 1].set_title('Rating vs Sales', fontweight='bold')
axes[1, 1].set_xlabel('Rating')
axes[1, 1].set_ylabel('Sales (₹)')
plt.colorbar(scatter, ax=axes[1, 1], label='Rating')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_chart(fig, '08_rating_analysis')

⭐ RATING ANALYSIS

Average Customer Rating: 3.99 / 5.0

--- Average Rating by Branch ---
Branch
A    3.84
B    3.98
C    4.05
D    4.09

--- Average Rating by Category ---
Category
Bakery           4.24
Snacks           4.11
Dairy            4.03
Vegetables       3.99
Beverages        3.96
Personal Care    3.95
Grocery          3.94
Fruits           3.83


  ✅ Saved: charts\08_rating_analysis.png


'charts\\08_rating_analysis.png'

---
## 12. EDA — Time / Monthly Trend Analysis

In [13]:
print('📅 MONTHLY TREND ANALYSIS\n')

# Monthly sales
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July']
monthly = df.groupby('Month_Name')['Sales'].agg(['sum', 'mean', 'count']).round(2)
monthly.columns = ['Total Sales', 'Avg Transaction', 'Transactions']
monthly = monthly.reindex(month_order)
print('--- Monthly Sales Summary ---')
print(monthly.to_string())

# --- Chart: Monthly Trends ---
fig, ax1 = plt.subplots(figsize=(14, 7))

# Line plot — total sales
color1 = COLORS['primary']
ax1.plot(monthly.index, monthly['Total Sales'], marker='o', linewidth=2.5,
         color=color1, markersize=8, markerfacecolor='white',
         markeredgecolor=color1, markeredgewidth=2, label='Total Sales', zorder=5)
ax1.fill_between(monthly.index, monthly['Total Sales'], alpha=0.15, color=color1)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Total Sales (₹)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Annotate values
for i, (month, row) in enumerate(monthly.iterrows()):
    ax1.annotate(f'₹{row["Total Sales"]:,.0f}', (month, row['Total Sales']),
                 textcoords="offset points", xytext=(0, 15), ha='center',
                 fontsize=9, fontweight='bold', color=color1)

# Secondary axis — transaction count
ax2 = ax1.twinx()
color2 = COLORS['accent']
ax2.bar(monthly.index, monthly['Transactions'], alpha=0.3, color=color2,
        edgecolor=color2, linewidth=1, label='Transactions', width=0.4)
ax2.set_ylabel('Number of Transactions', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('Monthly Sales Trend (Jan–Jul 2026)', fontsize=16, fontweight='bold', pad=15)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
plt.tight_layout()
save_chart(fig, '09_monthly_trend')

# --- Chart: Day of Week Analysis ---
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df.groupby('Day_of_Week')['Sales'].agg(['sum', 'count']).reindex(day_order)
daily.columns = ['Total Sales', 'Transactions']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(daily.index, daily['Total Sales'], color=sns.color_palette('husl', 7),
              edgecolor='white', linewidth=1.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'₹{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Sales by Day of Week', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Total Sales (₹)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_chart(fig, '10_day_of_week')

📅 MONTHLY TREND ANALYSIS

--- Monthly Sales Summary ---
            Total Sales  Avg Transaction  Transactions
Month_Name                                            
January        43415.53           493.36            88
February       30068.15           462.59            65
March          37306.42           449.47            83
April          52569.77           584.11            90
May            42542.31           500.50            85
June           35041.46           427.33            82
July            3467.44           495.35             7


  ✅ Saved: charts\09_monthly_trend.png
  ✅ Saved: charts\10_day_of_week.png


'charts\\10_day_of_week.png'

---
## 13. Correlation Analysis

In [14]:
print('🔗 CORRELATION ANALYSIS\n')

# Correlation matrix for numeric columns
numeric_cols = ['Quantity', 'Unit Price', 'Rating', 'Sales']
corr_matrix = df[numeric_cols].corr().round(3)
print('--- Correlation Matrix ---')
print(corr_matrix.to_string())

# --- Chart: Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=2, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
            annot_kws={'size': 13, 'fontweight': 'bold'},
            ax=ax)
ax.set_title('Correlation Heatmap — Numeric Variables', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
save_chart(fig, '11_correlation_heatmap')

🔗 CORRELATION ANALYSIS

--- Correlation Matrix ---
            Quantity  Unit Price  Rating  Sales
Quantity       1.000      -0.012   0.001  0.573
Unit Price    -0.012       1.000  -0.089  0.722
Rating         0.001      -0.089   1.000 -0.047
Sales          0.573       0.722  -0.047  1.000


  ✅ Saved: charts\11_correlation_heatmap.png


'charts\\11_correlation_heatmap.png'

---
## 14. Advanced Analysis — Branch × Category Heatmap

In [15]:
# Branch × Category sales heatmap
branch_cat = df.pivot_table(values='Sales', index='Category', columns='Branch',
                             aggfunc='sum').round(2)
print('--- Branch × Category Sales Matrix ---')
print(branch_cat.to_string())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(branch_cat, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=2, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Sales (₹)'},
            annot_kws={'size': 11, 'fontweight': 'bold'}, ax=ax)
ax.set_title('Sales Heatmap: Branch × Category', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Branch')
ax.set_ylabel('Category')
plt.tight_layout()
save_chart(fig, '12_branch_category_heatmap')

--- Branch × Category Sales Matrix ---
Branch                A         B         C         D
Category                                             
Bakery          1860.45   1990.01   1130.45   1535.19
Beverages       7934.12  18218.06  11637.59  18318.47
Dairy          10674.63  10516.22  16203.23   6597.92
Fruits          4986.25   8625.88   7306.09   2344.95
Grocery         9277.61   6750.82  13254.70  11187.34
Personal Care  12119.24  11425.62  15312.00   7087.10
Snacks          2994.70   3564.92   4803.03   5630.32
Vegetables      2510.08   3024.73   2822.36   2767.00


  ✅ Saved: charts\12_branch_category_heatmap.png


'charts\\12_branch_category_heatmap.png'

---
## 15. Top & Bottom Performers Summary

In [16]:
print('\n' + '='*60)
print('   📊 KEY FINDINGS SUMMARY')
print('='*60)

# Top product
print(f'\n🏆 Highest-Selling Product : {product_sales.index[0]} — ₹{product_sales.iloc[0]:,.2f}')
print(f'📉 Lowest-Selling Product  : {product_sales.index[-1]} — ₹{product_sales.iloc[-1]:,.2f}')

# Best branch
best_b = df.groupby(['Branch', 'City'])['Sales'].sum().sort_values(ascending=False)
print(f'\n🏆 Best Branch : {best_b.index[0][0]} ({best_b.index[0][1]}) — ₹{best_b.iloc[0]:,.2f}')

# Top category
print(f'\n🏆 Top Category : {cat_sales.index[0]} — ₹{cat_sales.iloc[0]["Total Sales"]:,.2f}')

# Most popular payment
print(f'\n🏆 Most Popular Payment : {payment_stats.index[0]} — {int(payment_stats.iloc[0]["Transactions"])} transactions')

# Member vs Normal
print(f'\n👥 Member Avg Transaction  : ₹{member_avg:,.2f}')
print(f'👥 Normal Avg Transaction  : ₹{normal_avg:,.2f}')

# Rating
print(f'\n⭐ Average Customer Rating : {avg_rating:.2f} / 5.0')


   📊 KEY FINDINGS SUMMARY

🏆 Highest-Selling Product : Cheese — ₹27,906.30
📉 Lowest-Selling Product  : Biscuits — ₹3,908.87

🏆 Best Branch : C (Mumbai) — ₹72,469.45

🏆 Top Category : Beverages — ₹56,108.24

🏆 Most Popular Payment : UPI — 127 transactions

👥 Member Avg Transaction  : ₹483.14
👥 Normal Avg Transaction  : ₹497.07

⭐ Average Customer Rating : 3.99 / 5.0


---
## 16. Business Recommendations

Based on the analysis, the following business recommendations are proposed:

1. **Stock Optimization:** Increase stock for high-selling products (Cheese, Cooking Oil, Coffee)
   and top categories (Beverages, Grocery).

2. **Branch Strategy:** Study Branch C (Mumbai) to understand why it outperforms other branches
   and replicate successful practices across all locations.

3. **Payment Infrastructure:** Continue investing in UPI payment infrastructure as it is the
   most popular payment method. Ensure smooth UPI transactions.

4. **Customer Programs:** Since Normal customers spend slightly more per transaction than Members,
   redesign membership benefits to incentivize higher spending among Members.

5. **Rating Improvement:** The average rating of 3.99/5 indicates room for improvement.
   Focus on customer service training and feedback mechanisms.

6. **Seasonal Planning:** Use monthly trend data to plan promotions and inventory for
   high-sales and low-sales months.

In [17]:
print('\n✅ Analysis Complete! All charts saved to the "charts/" folder.')
print(f'   Total charts generated: {len(os.listdir(CHART_DIR))}')
print(f'   Charts: {sorted(os.listdir(CHART_DIR))}')


✅ Analysis Complete! All charts saved to the "charts/" folder.
   Total charts generated: 12
   Charts: ['01_product_sales.png', '02_product_avg_sales.png', '03_branch_comparison.png', '04_category_analysis.png', '05_payment_analysis.png', '06_customer_type_analysis.png', '07_gender_analysis.png', '08_rating_analysis.png', '09_monthly_trend.png', '10_day_of_week.png', '11_correlation_heatmap.png', '12_branch_category_heatmap.png']
